# This notebook gives the stats on processed samples, ASVs generated, and taxa assigned for each study

Load master metadata excel file

In [19]:
import pandas as pd

metadata_file = "FReDNA_master_metadata.xlsx"
master_meta_df = pd.read_excel(metadata_file)
master_meta_df.shape

(351, 48)

dropped columns that are unnamed (i.e., empty)

In [20]:
unnamed_cols = master_meta_df.columns[master_meta_df.columns.str.startswith("Unnamed")]
meta_df_no_unnamed = master_meta_df.drop(unnamed_cols, axis=1)
meta_df_no_unnamed.shape

(351, 45)

subset for target study

In [21]:
target = "JuneJulyTemporal"

subset_meta_df = meta_df_no_unnamed[~meta_df_no_unnamed[target].isna()]
subset_meta_df.to_csv(target + "_metadata_sample_level.tsv", sep="\t", index=False)
n_total_entries_in_metadata = subset_meta_df.shape[0]
print("Number of entries in metadata:", n_total_entries_in_metadata)

Number of entries in metadata: 42


show how many unique entries there are

In [22]:
sample_id_col = subset_meta_df["Sample ID"]
n_unique_entries = sample_id_col.unique().shape[0]
print("Number of unqiue entries in metadata:", n_unique_entries)

Number of unqiue entries in metadata: 37


show duplicate sample entries

In [23]:
dupes = sample_id_col[sample_id_col.duplicated()]
n_dupes = len(dupes)
print("Number of duplicate entires in metadata:", f"{n_dupes}/{n_total_entries_in_metadata}")
print("Duplicate sample IDs:", end=" ")
for d in list(dupes):
    print(d, end=" ")
print()

Number of duplicate entires in metadata: 5/42
Duplicate sample IDs: SCD-UPL DTF-02R1 DTF-02R2 DTFA-01L1 DTFA-01L2 


get replicate-level metadata

In [36]:
from pathlib import Path

data_dir = Path("Aug_20_runs")
study_dir = list(data_dir.glob(f"{target}*"))[0]
rep_metadata = study_dir / f"{target}_metadata.tsv" # get the first (and hopefully only) entry
rep_meta_df = pd.read_csv(rep_metadata, sep="\t")
n_replicates = rep_meta_df.shape[0]
print("Number of sample replicates processed:", n_replicates)

Number of sample replicates processed: 159


see how many unique samples in metadata

In [37]:
unique_sams = rep_meta_df["Sample ID"].unique()
n_samples_processed = unique_sams.shape[0]
print("Number of samples processed:", f"{n_samples_processed}/{n_unique_entries}")

Number of samples processed: 18/37


record which samples have been processed and which haven't

In [28]:
sample_checklist = pd.DataFrame()
processed = []
unproccessed_sams = []

all_target_sams = subset_meta_df["Sample ID"].unique()
for sam in all_target_sams:
    if sam in unique_sams:
        processed.append("Y")
    else:
        unproccessed_sams.append(sam)
        processed.append("N")

sample_checklist["Sample ID"] = all_target_sams
sample_checklist["Processed"] = processed

sample_checklist.to_csv(f"{target}_checklist.tsv", sep="\t", index=False)
print(sample_checklist.shape)
sample_checklist.head()

(37, 2)


,Sample ID,Processed
0,DTG-05L,N
1,DTG-05R,N
2,DTFA-01L1,N
3,DTFA-01L2,N
4,DTF-02R1,N


take a look at unprocessed sams

In [29]:
unproc_sam_metadata = subset_meta_df[sample_id_col.isin(unproccessed_sams)]

n_unique_unproc_sams = subset_meta_df[sample_id_col.isin(unproccessed_sams)]["Sample ID"].unique().shape[0]

unproc_sam_metadata.to_csv(f"{target}_unprocessed.tsv", sep="\t", index=False)
n_unproc_samples = unproc_sam_metadata.shape[0]
print("Number of unprocessed samples:", f"{n_unique_unproc_sams}/{n_unique_entries}")

Number of unprocessed samples: 33/37


how many ASVs were assigned by the three classification steps?

In [30]:
import os

assigned_asvs = study_dir / "mapping_files" / "final_tax.tsv"
assigned_df = pd.read_csv(assigned_asvs, sep="\t")
assigned_df.shape

(465, 4)

how many ASVs were left unassigned?

In [31]:
unassigned_asvs = study_dir / "blast" / "blast_unassigned_tax.tsv"
unassigned_df = pd.read_csv(unassigned_asvs, sep="\t")
unassigned_df.shape

(112, 3)

what are the stats on assignement?

In [32]:
a_len = assigned_df.shape[0]
u_len = unassigned_df.shape[0]
total_asvs = a_len + u_len
print(f"Assigned ASVs: {a_len} / {total_asvs} ({round(round(a_len / total_asvs, 2)*100)}%)")
print(f"Unassigned ASVs: {u_len} / {total_asvs} ({round(round(u_len / total_asvs, 2)*100)}%)")

Assigned ASVs: 465 / 577 (81%)
Unassigned ASVs: 112 / 577 (19%)


get the number of unqiue taxa

In [33]:
final_feat_tab = study_dir / "mapping_files" / "feat_tab_unique.tsv"
final_ftab_df = pd.read_csv(final_feat_tab, sep="\t")
n_unique_taxa = final_ftab_df.shape[0]
print("Number of unique taxa:", n_unique_taxa)

Number of unique taxa: 130


look at the taxa with the most total reads

In [ ]:
final_ftab_df.index = final_ftab_df["Taxon"]
fftab_taxon = final_ftab_df.drop("Taxon", axis=1)
top_ten_taxa = list(fftab_taxon.sum(axis=1).sort_values(ascending=False).index[:10])
print("Top 10 taxa with the most gross read counts:")
i = 0
for t in top_ten_taxa:
    i += 1
    print(f"\t{i}. {t}")

Top 10 taxa with the most gross read counts:
	1. Carassius auratus
	2. Dorosoma cepedianum
	3. Micropterus dolomieu
	4. Lepomis macrochirus
	5. Cyprinella
	6. Catostomidae
	7. Anas
	8. Fundulus notatus
	9. Leuciscidae
	10. Etheostoma flabellare
